# 📊 Exploratory Data Analysis: Kermany Chest X-Ray Dataset (Pneumonia)

> **Dataset Reference:** Kermany et al. 2018 (Guangzhou Women and Children's Medical Center)  
> **Source:** Kaggle `paultimothymooney/chest-xray-pneumonia`  
> **License:** Creative Commons Attribution 4.0 International (CC BY 4.0)

---

## 🎯 Purpose & Dataset Usage in Graduation Thesis

1. **Core Training Source:** Primary data source for 3 major classes: `Normal`, `Bacterial Pneumonia`, and `Viral Pneumonia`.
2. **Clinical Narrative:** Represents **pediatric chest X-rays** (patients aged 1 to 5 years). Distinguishing bacterial from viral pneumonia is crucial in pediatric care to prevent unnecessary antibiotic prescriptions.
3. **Sub-label Inference:** Filenames contain `bacteria` or `virus` substrings, allowing 3-class granularity rather than binary classification.


In [ ]:
import os
import re
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Plotting settings
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path("../data/raw/kermany")
print(f"Data directory exists: {DATA_DIR.exists()}")


## 1. File Scanning & Metadata Extraction

We scan all `.jpeg` images, exclude macOS metadata (`__MACOSX` / `._*`), and parse:
- Class label (`normal`, `bacterial_pneumonia`, `viral_pneumonia`)
- Patient ID derived from `person{N}` filename pattern


In [ ]:
raw_images = [
    p for p in DATA_DIR.rglob("*.[jJ][pP]*[gG]")
    if "__MACOSX" not in str(p) and not p.name.startswith("._")
]

records = []
for p in raw_images:
    fn = p.name
    parent = p.parent.name.upper()
    
    if "NORMAL" in parent or "IM-" in fn:
        label = "normal"
        patient_id = f"kermany_norm_{p.stem}"
    elif "PNEUMONIA" in parent or "bacteria" in fn or "virus" in fn:
        if "bacteria" in fn.lower():
            label = "bacterial_pneumonia"
        elif "virus" in fn.lower():
            label = "viral_pneumonia"
        else:
            label = "bacterial_pneumonia"
        
        m = re.search(r"person(\d+)", fn, re.IGNORECASE)
        patient_id = f"person{m.group(1)}" if m else f"kermany_pneu_{p.stem}"
    else:
        continue
        
    records.append({
        "filepath": str(p),
        "filename": fn,
        "label": label,
        "patient_id": patient_id
    })

df = pd.DataFrame(records)
print(f"Total scanned images: {len(df)}")
print(f"Unique patient groups: {df['patient_id'].nunique()}")
df.head()


## 2. Class Distribution Analysis

Let's visualize the balance between `Normal`, `Bacterial Pneumonia`, and `Viral Pneumonia`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart
class_counts = df['label'].value_counts()
colors = ['#2b5c8f', '#d95f02', '#7570b3']
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], hue=class_counts.index, palette=colors, legend=False)
axes[0].set_title("Image Count by Class", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Number of Images")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie Chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.03, 0.03, 0.03))
axes[1].set_title("Class Proportions", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## 3. Image Dimensions & Aspect Ratio Analysis

Medical X-rays come in varying resolutions. We inspect width, height, and aspect ratios.


In [ ]:
widths, heights, aspect_ratios = [], [], []

for p in df['filepath'].sample(min(1000, len(df)), random_state=42):
    with Image.open(p) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(w / h)

df_dim = pd.DataFrame({"width": widths, "height": heights, "aspect_ratio": aspect_ratios})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df_dim['width'], ax=axes[0], color='#2b5c8f', kde=True)
axes[0].set_title("Image Width Distribution")

sns.histplot(df_dim['height'], ax=axes[1], color='#d95f02', kde=True)
axes[1].set_title("Image Height Distribution")

sns.histplot(df_dim['aspect_ratio'], ax=axes[2], color='#7570b3', kde=True)
axes[2].set_title("Aspect Ratio (W/H)")
axes[2].axvline(1.0, color='red', linestyle='--', label='Square (1.0)')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Width - Min: {min(widths)}, Max: {max(widths)}, Median: {np.median(widths):.0f}")
print(f"Height - Min: {min(heights)}, Max: {max(heights)}, Median: {np.median(heights):.0f}")


## 4. Visualizing X-Ray Images per Class

We display representative chest X-rays from each class.
- **Normal:** Clear lung fields, sharp diaphragm boundaries.
- **Bacterial Pneumonia:** Focal/lobar consolidation (dense white opacities in specific lung lobes).
- **Viral Pneumonia:** Diffuse interstitial patterns (bilateral patchiness).


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row_idx, label in enumerate(['normal', 'bacterial_pneumonia', 'viral_pneumonia']):
    samples = df[df['label'] == label].sample(4, random_state=42).reset_index()
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        img_path = samples.iloc[col_idx]['filepath']
        img = Image.open(img_path).convert('L')
        ax.imshow(img, cmap='gray')
        pid = samples.iloc[col_idx]['patient_id']
        ax.set_title(f"{label.upper()} \n Patient: {pid}", fontsize=10)
        ax.axis('off')

plt.suptitle("Representative Pediatric Chest X-Rays by Diagnosis", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 5. Pixel Intensity & Mean Heatmap Analysis

Comparing pixel intensity distributions across classes.


In [ ]:
plt.figure(figsize=(10, 5))

for label, color in zip(['normal', 'bacterial_pneumonia', 'viral_pneumonia'], ['#2b5c8f', '#d95f02', '#7570b3']):
    sample_paths = df[df['label'] == label]['filepath'].sample(200, random_state=42)
    pixels = []
    for p in sample_paths:
        img = np.array(Image.open(p).convert('L').resize((224, 224)))
        pixels.extend(img.flatten())
    sns.kdeplot(pixels, label=label, color=color, linewidth=2)

plt.title("Pixel Intensity Histogram by Class (Resized 224x224)", fontsize=14, fontweight='bold')
plt.xlabel("Pixel Value (0-255)")
plt.ylabel("Density")
plt.legend()
plt.show()


## 📌 Summary & Key Takeaways for Project Execution

| Aspect | Observation / Technical Insight |
| :--- | :--- |
| **Data Imbalance** | Bacterial Pneumonia is the majority class (~47%), Normal (~27%), Viral (~25%). Requires class weighting / WeightedRandomSampler. |
| **Patient Leakage** | Patient IDs follow `person{N}`. Split logic MUST group by `patient_id` so same child never appears in train and test. |
| **Anatomical Domain** | Pediatric dataset (1-5 yrs old) -> Smaller thoracic cavity, heart occupies larger ratio of chest than adults. |
| **Pre-processing** | All images resized to `224x224` & normalized to ImageNet mean/std for transfer learning backbones (`DenseNet-121`, `ResNet-18`). |
